# Getting Started with Grid Volatility Data

This notebook shows you how to load the data pack using both **SQLite** and **DuckDB**.

- **SQLite** is built into Python (`import sqlite3`). Zero setup!
- **DuckDB** is installed via `pip`. Also zero setup, and is optimised for analytical queries.

Choose whichever you're comfortable with. Both work equally well for this challenge.

In [ ]:
import sqlite3
import duckdb
import pandas as pd
import matplotlib.pyplot as plt

# Path to the data pack
CSV_PATH = "../data/grid_volatility_expanded_dataset.csv"

## Option 1: Using SQLite

SQLite is built into Python. No installation needed.

In [ ]:
# Connect to SQLite (creates an in-memory database)
con = sqlite3.connect(":memory:")

# Load the CSV into a table
df = pd.read_csv(CSV_PATH)
df.to_sql("grid_volatility", con, index=False, if_exists="replace")

# Query it
result = pd.read_sql_query("SELECT * FROM grid_volatility LIMIT 5", con)
result

In [ ]:
# Summary statistics
pd.read_sql_query("""
    SELECT 
        COUNT(*) as row_count,
        MIN(timestamp) as earliest,
        MAX(timestamp) as latest,
        AVG(power_demand_mw) as avg_demand
    FROM grid_volatility
""", con)

## Option 2: Using DuckDB

DuckDB can query the CSV directly — no loading step required. This is faster for large datasets and analytical queries.

In [ ]:
# Connect to DuckDB (in-memory)
duck = duckdb.connect()

# Query the CSV directly
duck.sql(f"SELECT * FROM '{CSV_PATH}' LIMIT 5").df()

In [ ]:
# More complex query: daily average demand
duck.sql(f"""
    SELECT 
        DATE_TRUNC('day', timestamp::TIMESTAMP) as day,
        AVG(power_demand_mw) as avg_demand
    FROM '{CSV_PATH}'
    GROUP BY 1
    ORDER BY 1
    LIMIT 10
""").df()

## Visualise Demand Over Time

In [ ]:
# Load and plot
df = duck.sql(f"""
    SELECT timestamp::TIMESTAMP as ts, power_demand_mw
    FROM '{CSV_PATH}'
    ORDER BY ts
""").df()

df.plot(x="ts", y="power_demand_mw", figsize=(14, 5), title="Electricity Demand Over Time")
plt.tight_layout()
plt.show()

## Next Steps

Now it's your turn. Consider:

1. **Explore the data**: What patterns do you see? Daily, weekly, seasonal?
2. **Identify weather drivers**: How does temperature relate to demand?
3. **Handle data issues**: What gaps, outliers, or duplicates exist? How will you address them?
4. **Form a hypothesis**: What drives demand volatility? Write it in `DECISION_MEMO.md`.
5. **Build a forecast**: Start simple
6. **Make a recommendation**: What should the trading desk do?